In [10]:
import ilpy
import motile
import numpy as np
import toml
import zarr
import geff
import networkx as nx
from typing import TYPE_CHECKING, Any, DefaultDict, Hashable, Iterator, cast

from mhat.tracking import utils
from mhat.tracking.tracks_io import save_tracks_to_csv
from motile_toolbox.candidate_graph import graph_to_nx

In [2]:
def add_cand_edges(
    cand_graph: nx.DiGraph,
    max_edge_distance: float,
) -> None:
    """Add candidate edges to a candidate graph by connecting all nodes in adjacent
    frames that are closer than max_edge_distance. Also adds attributes to the edges.

    Args:
        cand_graph (nx.DiGraph): Candidate graph with only nodes populated. Will
            be modified in-place to add edges.
        max_edge_distance (float): Maximum distance that objects can travel between
            frames. All nodes within this distance in adjacent frames will by connected
            with a candidate edge.
        node_frame_dict (dict[int, list[Any]] | None, optional): A mapping from frames
            to node ids. If not provided, it will be computed from cand_graph. Defaults
            to None.
    """
    node_frame_dict = utils._compute_node_frame_dict(cand_graph)
    print(f"Node frame dict: {node_frame_dict}")

    frames = sorted(node_frame_dict.keys())
    prev_node_ids = node_frame_dict[frames[0]]
    prev_kdtree = utils.create_kdtree(cand_graph, prev_node_ids)
    print(f"Prev kdtree data: {prev_kdtree.data}")
    for frame in frames:
        if frame + 1 not in node_frame_dict:
            continue
        next_node_ids = node_frame_dict[frame + 1]
        next_kdtree = utils.create_kdtree(cand_graph, next_node_ids)
        print(f"Next kdtree data: {next_kdtree.data}")

        # match indices based on a max edge distance
        #matched_indices = prev_kdtree.query_ball_tree(next_kdtree, max_edge_distance)

        # match indices based on k nearest neighbors
        _, matched_indices = next_kdtree.query(prev_kdtree.data, k=2)

        print(f"Matched indices: {matched_indices}")

        for prev_node_id, next_node_indices in zip(prev_node_ids, matched_indices):
            for next_node_index in next_node_indices:
                next_node_id = next_node_ids[next_node_index]
                cand_graph.add_edge(prev_node_id, next_node_id)

        prev_node_ids = next_node_ids
        prev_kdtree = next_kdtree

In [11]:
def to_nx_graph(graph, flatten_hyperedges: bool = True) -> nx.DiGraph:
    """Convert a this TrackGraph into a networkx DiGraph.
    Args:
        flatten_hyperedges (bool, optional): If True, include one edge for each
            (source, target) combo in a hyperedge. If False, introduce a new
            hypernode to represent hyperedges. Defaults to True.
    Returns:
        networkx.DiGraph: Directed networkx graph with same nodes, edges, and
            attributes.
    """
    nx_graph = nx.DiGraph()
    nodes_list = list(graph.nodes.items())
    nx_graph.add_nodes_from(nodes_list)
    edges_list: list[tuple[Any, Any, Mapping]] = []
    for edge, data in graph.edges.items():
        if graph.is_hyperedge(edge):
            us, vs = edge
            if flatten_hyperedges:
                # flatten the hyperedges into multiple normal edges
                for u in us:
                    for v in vs:
                        edges_list.append((u, v, {}))
            else:
                # add a hypernode to connect all in nodes with all out nodes
                hypernode_id = "_".join(list(map(str, us)) + list(map(str, vs)))
                nx_graph.add_node(hypernode_id, **data)
                for u in us:
                    edges_list.append((u, hypernode_id, {}))
                for v in vs:
                    edges_list.append((hypernode_id, v, {}))
        else:
            u, v = edge  # type: ignore
            edges_list.append((u, v, data))

    nx_graph.add_edges_from(edges_list)
    return nx_graph

In [12]:
def solve_with_motile(config, graph, exclusion_sets):
    """Set up and solve the network flow problem.

    Args:
        graph (motile.TrackGraph): The candidate graph.

    Returns:
        nx.DiGraph: The networkx digraph with the selected solution tracks
    """
    solver = motile.Solver(graph)

    solver.add_constraint(motile.constraints.MaxParents(1))
    solver.add_constraint(motile.constraints.MaxChildren(1))

    solver.add_cost(
        motile.costs.EdgeSelection(
            weight=config["drift_weight"],
            attribute="drift_dist",
            constant=config["drift_constant"],
        ),
        name="drift",
    )

    solver.add_cost(
        motile.costs.EdgeSelection(
            weight=config["area_weight"],
            attribute="area_diff",
            constant=config["area_constant"],
        ),
        name="area",
    )

    solver.add_cost(
        motile.costs.NodeSelection(
            weight=config["cohesion_weight"],
            attribute="cohesion",
            constant=config["cohesion_constant"],
        ),
        name="cohesion",
    )

    solver.add_cost(
        motile.costs.NodeSelection(
            weight=config["adhesion_weight"],
            attribute="adhesion",
            constant=config["adhesion_constant"],
        ),
        name="adhesion",
    )

    solver.add_cost(
        motile.costs.Appear(
            constant=config["appear_constant"], ignore_attribute="ignore_appear"
        )
    )
    solver.add_cost(
        motile.costs.Disappear(
            constant=config["disappear_constant"], ignore_attribute="ignore_disappear"
        )
    )

    solver.add_constraint(motile.constraints.ExclusiveNodes(exclusion_sets))

    solver.solve()
    # switch to new function
    solution_graph = to_nx_graph(solver.get_selected_subgraph())
    return solution_graph

In [20]:
cand_graph = nx.DiGraph()

cand_graph.add_node(1, time=0, label=1, x=10, y=10, z=10, area=6, cohesion=0.5, adhesion=0.5)
cand_graph.add_node(2, time=1, label=2, x=8, y=10, z=10, area=3, cohesion=0.5, adhesion=0.5)
cand_graph.add_node(3, time=1, label=3, x=12, y=10, z=10, area=3, cohesion=0.5, adhesion=0.5)
cand_graph.add_node(4, time=0, label=4, x=30, y=30, z=30, area=6, cohesion=0.5, adhesion=0.5)
cand_graph.add_node(5, time=1, label=5, x=28, y=30, z=30, area=6, cohesion=0.5, adhesion=0.5)

exclusion_sets = []

cand_graph.nodes(data=True)

NodeDataView({1: {'time': 0, 'label': 1, 'x': 10, 'y': 10, 'z': 10, 'area': 6, 'cohesion': 0.5, 'adhesion': 0.5}, 2: {'time': 1, 'label': 2, 'x': 8, 'y': 10, 'z': 10, 'area': 3, 'cohesion': 0.5, 'adhesion': 0.5}, 3: {'time': 1, 'label': 3, 'x': 12, 'y': 10, 'z': 10, 'area': 3, 'cohesion': 0.5, 'adhesion': 0.5}, 4: {'time': 0, 'label': 4, 'x': 30, 'y': 30, 'z': 30, 'area': 6, 'cohesion': 0.5, 'adhesion': 0.5}, 5: {'time': 1, 'label': 5, 'x': 28, 'y': 30, 'z': 30, 'area': 6, 'cohesion': 0.5, 'adhesion': 0.5}})

In [21]:
add_cand_edges(cand_graph, max_edge_distance=5)
print("Edges before hyperedges: ", cand_graph.number_of_edges())
cand_graph = utils.add_division_hyperedges(cand_graph)
# TODO: add merge hyperedges and divisions/merges involving more than two nodes
print("Edges after hyperedges: ", cand_graph.number_of_edges())

Node frame dict: {0: [1, 4], 1: [2, 3, 5]}
Prev kdtree data: [[10. 10. 10.]
 [30. 30. 30.]]
Next kdtree data: [[ 8. 10. 10.]
 [12. 10. 10.]
 [28. 30. 30.]]
Matched indices: [[1 0]
 [2 1]]
Edges before hyperedges:  4
Edges after hyperedges:  10


In [22]:
print("Nodes: ")
print(cand_graph.nodes(data=True))
print("Edges: ")
print(cand_graph.edges(data=True))

Nodes: 
[(1, {'time': 0, 'label': 1, 'x': 10, 'y': 10, 'z': 10, 'area': 6, 'cohesion': 0.5, 'adhesion': 0.5}), (2, {'time': 1, 'label': 2, 'x': 8, 'y': 10, 'z': 10, 'area': 3, 'cohesion': 0.5, 'adhesion': 0.5}), (3, {'time': 1, 'label': 3, 'x': 12, 'y': 10, 'z': 10, 'area': 3, 'cohesion': 0.5, 'adhesion': 0.5}), (4, {'time': 0, 'label': 4, 'x': 30, 'y': 30, 'z': 30, 'area': 6, 'cohesion': 0.5, 'adhesion': 0.5}), (5, {'time': 1, 'label': 5, 'x': 28, 'y': 30, 'z': 30, 'area': 6, 'cohesion': 0.5, 'adhesion': 0.5}), ('1_3_2', {}), ('4_5_3', {})]
Edges: 
[(1, 3, {}), (1, 2, {}), (1, '1_3_2', {}), (4, 5, {}), (4, 3, {}), (4, '4_5_3', {}), ('1_3_2', 3, {}), ('1_3_2', 2, {}), ('4_5_3', 5, {}), ('4_5_3', 3, {})]


In [23]:
utils.add_appear_ignore_attr(cand_graph)
utils.add_disappear(cand_graph)
track_graph = motile.TrackGraph(cand_graph, frame_attribute="time")
utils.add_drift_dist_attr(track_graph, drift=0)
utils.add_area_diff_attr(track_graph)
print("Track graph nodes: ")
print(track_graph.nodes)
print("Track graph edges: ")
print(track_graph.edges)

Track graph nodes: 
{1: {'time': 0, 'label': 1, 'x': 10, 'y': 10, 'z': 10, 'area': 6, 'cohesion': 0.5, 'adhesion': 0.5, 'ignore_appear': True}, 2: {'time': 1, 'label': 2, 'x': 8, 'y': 10, 'z': 10, 'area': 3, 'cohesion': 0.5, 'adhesion': 0.5}, 3: {'time': 1, 'label': 3, 'x': 12, 'y': 10, 'z': 10, 'area': 3, 'cohesion': 0.5, 'adhesion': 0.5}, 4: {'time': 0, 'label': 4, 'x': 30, 'y': 30, 'z': 30, 'area': 6, 'cohesion': 0.5, 'adhesion': 0.5, 'ignore_appear': True}, 5: {'time': 1, 'label': 5, 'x': 28, 'y': 30, 'z': 30, 'area': 6, 'cohesion': 0.5, 'adhesion': 0.5}}
Track graph edges: 
{(1, 3): {'drift_dist': 2, 'area_diff': np.int64(3)}, (1, 2): {'drift_dist': 2, 'area_diff': np.int64(3)}, ((1,), (3, 2)): {'drift_dist': 0.0, 'area_diff': np.int64(0)}, (4, 5): {'drift_dist': 2, 'area_diff': np.int64(0)}, (4, 3): {'drift_dist': 18, 'area_diff': np.int64(3)}, ((4,), (5, 3)): {'drift_dist': 10.0, 'area_diff': np.int64(3)}}


In [24]:
config = {
    "drift": 0,
    "drift_weight": 1.0,
    "drift_constant": -10.0,
    "area_weight": 0.5,
    "area_constant": 0.0,
    "cohesion_weight": 10.0,
    "cohesion_constant": -500.0,
    "adhesion_weight": -100.0,
    "adhesion_constant": 0.0,
    "appear_constant": 200.0,
    "disappear_constant": 200.0,
}

solution_graph = solve_with_motile(config, track_graph, exclusion_sets)

Set parameter NonConvex to value 2
Set parameter Threads to value 1


In [25]:
print("Solution graph nodes: ")
print(solution_graph.nodes)
print("Solution graph edges: ")
print(solution_graph.edges)

Solution graph nodes: 
[1, 2, 3, 4, 5]
Solution graph edges: 
[(1, 3), (1, 2), (4, 5)]
